In [1]:
# ─────────────────────────────────────────────
# CELL 1 — Install & Imports
# ─────────────────────────────────────────────
!pip install -q transformers==4.40.2 accelerate sentencepiece

import torch
import pandas as pd
import re
from tqdm.auto import tqdm
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM

print("All imports done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 3.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 72.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 67.6 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
All imports done.


In [2]:
# ─────────────────────────────────────────────
# CELL 2 — Load Phi-3-mini
# ─────────────────────────────────────────────
model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",          # ← changed from {"": 0} to "auto"
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation="eager" # ← add this: disables flash-attn which caused the warning
)
model.eval()

NUM_LAYERS  = model.config.num_hidden_layers   # 32
HIDDEN_SIZE = model.config.hidden_size          # 3072
print(f"num_layers  = {NUM_LAYERS}")
print(f"hidden_size = {HIDDEN_SIZE}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

num_layers  = 32
hidden_size = 3072


In [3]:
# ─────────────────────────────────────────────
# CELL 3 — Load your prompt CSV
# ─────────────────────────────────────────────
PROMPT_CSV = "/kaggle/input/datasets/aparnakrishna2407/phi3-experiment/phi3_experiment_final.csv"

prompt_df = pd.read_csv(PROMPT_CSV)
print("Shape:", prompt_df.shape)
print("Columns:")
for c in prompt_df.columns:
    print(" ", c)

Shape: (80, 149)
Columns:
  userId
  gender
  age_group
  itemIds
  artistIds
  prompt_counterfact-False_sample-random_userDemo-no_info_interact-zero-shot
  prompt_counterfact-False_sample-random_userDemo-no_info_interact-ICL-Few-shot-1
  prompt_counterfact-False_sample-random_userDemo-no_info_interact-ICL-Few-shot-2
  prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot
  prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-1
  prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-2
  prompt_counterfact-False_sample-random_userDemo-age-group_interact-zero-shot
  prompt_counterfact-False_sample-random_userDemo-age-group_interact-ICL-Few-shot-1
  prompt_counterfact-False_sample-random_userDemo-age-group_interact-ICL-Few-shot-2
  prompt_counterfact-False_sample-random_userDemo-intersectional_interact-zero-shot
  prompt_counterfact-False_sample-random_userDemo-intersectional_interact-ICL-Few-shot-1
  prompt_counterfact-Fa

In [4]:
# ─────────────────────────────────────────────
# CELL 4 — Helper: extract Phi-3 hidden states
# ─────────────────────────────────────────────
def get_hidden_states(prompt_text):
    """
    Forward-pass a prompt through Phi-3 with output_hidden_states=True.
    Returns dict {layer_idx (0-indexed): tensor [seq_len, 3072]}
    """
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # outputs.hidden_states is a tuple of length (num_layers + 1)
    # index 0 = embedding layer, 1..32 = transformer layers
    hs_by_layer = {}
    for layer_idx in range(NUM_LAYERS):
        # squeeze batch dim → [seq_len, 3072], move to cpu as float32
        hs_by_layer[layer_idx] = outputs.hidden_states[layer_idx + 1].squeeze(0).cpu().float()

    return hs_by_layer

print("get_hidden_states defined.")

get_hidden_states defined.


In [5]:
test = [col for col in prompt_df.columns if "userDemo-gender" in col and "counterfact-False" in col]
print(len(test), "columns found")
for c in test: print(c)

18 columns found
prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot
prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-1
prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-2
prompt_counterfact-False_sample-frequent_userDemo-gender_interact-zero-shot
prompt_counterfact-False_sample-frequent_userDemo-gender_interact-ICL-Few-shot-1
prompt_counterfact-False_sample-frequent_userDemo-gender_interact-ICL-Few-shot-2
prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-zero-shot
prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-ICL-Few-shot-1
prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-ICL-Few-shot-2
recommendation_prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot
recommendation_prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-1
recommendation_prompt_counterfact-False_sample-random_userDemo-gender_i

In [6]:
# ─────────────────────────────────────────────
# CELL 5 — Build gender-contrastive prompt pairs
# ─────────────────────────────────────────────

GENDER_COL = "prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-zero-shot"

# Verify the column exists
assert GENDER_COL in prompt_df.columns, f"Column not found: {GENDER_COL}"

def make_male_version(text):
    t = str(text)
    t = re.sub(r'\b[Ff]emale\b', 'male', t)
    t = re.sub(r'\b[Ww]oman\b',  'man',  t)
    t = re.sub(r'\b[Ww]omen\b',  'men',  t)
    t = re.sub(r'\b[Ss]he\b',    'he',   t)
    t = re.sub(r'\b[Hh]er\b',    'his',  t)
    return t

def make_female_version(text):
    t = str(text)
    t = re.sub(r'\b[Mm]ale\b',  'female', t)
    t = re.sub(r'\b[Mm]an\b',   'woman',  t)
    t = re.sub(r'\b[Mm]en\b',   'women',  t)
    t = re.sub(r'\b[Hh]e\b',    'she',    t)
    t = re.sub(r'\b[Hh]is\b',   'her',    t)
    return t

NUM_SV_PAIRS = 30
source_prompts = prompt_df[GENDER_COL].dropna().head(NUM_SV_PAIRS).tolist()
prompt_pairs = [(make_male_version(p), make_female_version(p)) for p in source_prompts]

# Sanity check
print("--- MALE version (first 300 chars) ---")
print(prompt_pairs[0][0][:300])
print("\n--- FEMALE version (first 300 chars) ---")
print(prompt_pairs[0][1][:300])
print(f"\nTotal pairs: {len(prompt_pairs)}")

--- MALE version (first 300 chars) ---
The user is male and Early Adult (≤24 yrs). The user has listenned to the following songs in the past, organized as (Song - Artist):

- "Gangsta Bop" by Akon
- "I Can'T Wait" by Akon
- "My Love" by Joe
- "One For Me" by Lloyd
- "Get'Cha Head In The Game (Pop Version)" by B5
- "The Backyardigans Them

--- FEMALE version (first 300 chars) ---
The user is Female and Early Adult (≤24 yrs). The user has listenned to the following songs in the past, organized as (Song - Artist):

- "Gangsta Bop" by Akon
- "I Can'T Wait" by Akon
- "My Love" by Joe
- "One For Me" by Lloyd
- "Get'Cha Head In The Game (Pop Version)" by B5
- "The Backyardigans Th

Total pairs: 30


In [7]:
# ─────────────────────────────────────────────
# CELL 6 — Compute Phi-3 gender steering vectors
#           (prompt_avg_diff method)
#
# Formula:  V_gender[layer] = mean_over_pairs( h_male[layer].mean(tokens)
#                                             - h_female[layer].mean(tokens) )
#
# This is the same method your professor used for Llama,
# now applied directly to Phi-3's own representation space.
# ─────────────────────────────────────────────
diffs_per_layer = defaultdict(list)   # layer_idx → list of [3072] tensors

print("Extracting hidden states from Phi-3...")
for male_p, female_p in tqdm(prompt_pairs, desc="Prompt pairs"):

    hs_male   = get_hidden_states(male_p)
    hs_female = get_hidden_states(female_p)

    for layer_idx in range(NUM_LAYERS):
        hm = hs_male[layer_idx]    # [seq_len, 3072]
        hf = hs_female[layer_idx]  # [seq_len, 3072]

        # Average over token positions → [3072]
        diff = hm.mean(dim=0) - hf.mean(dim=0)
        diffs_per_layer[layer_idx].append(diff)

# Average across all pairs → one steering vector per layer
phi3_sv_gender = {
    layer_idx: torch.stack(vecs).mean(dim=0)
    for layer_idx, vecs in diffs_per_layer.items()
}

print(f"\nDone. Vectors computed for {len(phi3_sv_gender)} layers.")
print(f"Shape at layer 0: {phi3_sv_gender[0].shape}")  # should be [3072]

# Save the steering vectors
SV_SAVE_PATH = "/kaggle/working/phi3_gender-bias_prompt_avg_diff.pt"
torch.save(phi3_sv_gender, SV_SAVE_PATH)
print(f"Saved steering vectors to: {SV_SAVE_PATH}")

Extracting hidden states from Phi-3...


Prompt pairs:   0%|          | 0/30 [00:00<?, ?it/s]

You are not running the flash-attention implementation, expect numerical differences.
2026-09-25 07:03:07.044456: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790319787.325251      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790319787.391808      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1790319788.042828      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790319788.042866      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W


Done. Vectors computed for 32 layers.
Shape at layer 0: torch.Size([3072])
Saved steering vectors to: /kaggle/working/phi3_gender-bias_prompt_avg_diff.pt


In [9]:
# ─────────────────────────────────────────────
# CELL 6B — Compute response-level steering vectors
#            (response_avg_diff method) — FIXED
# ─────────────────────────────────────────────

response_diffs_per_layer = defaultdict(list)

print("Extracting response-level hidden states from Phi-3...")
print("(Uses 10 pairs — slower per pair but fewer needed)")

for male_p, female_p in tqdm(prompt_pairs[:10], desc="Response pairs"):
    for gender_prompt, sign in [(male_p, +1), (female_p, -1)]:

        inputs = tokenizer(
            gender_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(model.device)

        with torch.no_grad():
            gen_out = model.generate(
                **inputs,
                max_new_tokens=30,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                return_dict_in_generate=True,
                output_hidden_states=True
            )

        # gen_out.hidden_states: tuple of length = num generated tokens
        # Each element: tuple of (num_layers+1) tensors
        # First token shape: [batch, prompt_len+1, 3072] — need only last position
        # Other tokens shape: [batch, 1, 3072]
        for step_idx, step_hs in enumerate(gen_out.hidden_states):
            for layer_idx in range(NUM_LAYERS):
                hs = step_hs[layer_idx + 1]          # [batch, seq, 3072]
                vec = hs[0, -1, :].cpu().float()     # always take LAST token → [3072]
                response_diffs_per_layer[layer_idx].append(vec * sign)

# Average across all steps and pairs
phi3_sv_gender_response = {
    layer_idx: torch.stack(vecs).mean(dim=0)
    for layer_idx, vecs in response_diffs_per_layer.items()
}

print(f"\nDone. Response vectors computed for {len(phi3_sv_gender_response)} layers.")
print(f"Shape at layer 0: {phi3_sv_gender_response[0].shape}")  # should be [3072]

torch.save(phi3_sv_gender_response,
           "/kaggle/working/phi3_gender-bias_response_avg_diff.pt")
print("Saved: /kaggle/working/phi3_gender-bias_response_avg_diff.pt")

Extracting response-level hidden states from Phi-3...
(Uses 10 pairs — slower per pair but fewer needed)


Response pairs:   0%|          | 0/10 [00:00<?, ?it/s]


Done. Response vectors computed for 32 layers.
Shape at layer 0: torch.Size([3072])
Saved: /kaggle/working/phi3_gender-bias_response_avg_diff.pt


In [10]:
# ─────────────────────────────────────────────
# CELL 7 — Define steering hook + generation functions
#
# The formula from your whiteboard:
#   h' = h + λ · V_gender
#
# We apply this at every layer in STEER_LAYERS during generation.
# Hooks are registered before generation and removed immediately after.
# ─────────────────────────────────────────────

# Steer middle layers (most effective for semantic content)
STEER_LAYERS = list(range(0, 32))   # all 32 layers

# Lambda values to sweep (negative = steer away from male direction = de-bias)
LAMBDA_VALUES = [-3.0, -2.0, -1.0, 0.0, 1.0]


def _apply_hook(module, inputs, output, steer_vec, lam):
    """
    Forward hook: adds λ·V_norm to hidden states.
    Normalising the vector makes λ a consistent, comparable scale.
    """
    hs = output[0] if isinstance(output, tuple) else output  # [batch, seq_len, 3072]
    v  = steer_vec.to(device=hs.device, dtype=hs.dtype)
    v  = v / (v.norm() + 1e-8)                               # unit vector
    hs = hs + lam * v.view(1, 1, -1)                         # broadcast over batch & seq
    if isinstance(output, tuple):
        return (hs,) + output[1:]
    return hs


def generate_steered(prompt_text, lam, sv_dict=None, max_new_tokens=60):
    """
    Generate with steering hooks active.
    lam=0.0 is equivalent to baseline (no steering effect after normalization,
    but we still call baseline separately for clarity).
    """
    if sv_dict is None:
        sv_dict = phi3_sv_gender

    messages = [{"role": "user", "content": str(prompt_text)}]
    text     = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(text, return_tensors="pt").to(model.device)

    # Register hooks on all steer layers
    handles = []
    for layer_idx in STEER_LAYERS:
        if layer_idx in sv_dict:
            vec = sv_dict[layer_idx]
            h   = model.model.layers[layer_idx].register_forward_hook(
                lambda m, inp, out, v=vec, l=lam: _apply_hook(m, inp, out, v, l)
            )
            handles.append(h)

    try:
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        for h in handles:       # ALWAYS remove hooks
            h.remove()

    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)


def generate_baseline(prompt_text, max_new_tokens=60):
    """Generate WITHOUT any steering — pure Phi-3 output."""
    messages = [{"role": "user", "content": str(prompt_text)}]
    text     = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)


print("Hook and generation functions defined.")
print(f"Steering layers: {STEER_LAYERS}")
print(f"Lambda values to sweep: {LAMBDA_VALUES}")

Hook and generation functions defined.
Steering layers: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
Lambda values to sweep: [-3.0, -2.0, -1.0, 0.0, 1.0]


In [11]:
# ─────────────────────────────────────────────
# CELL 8 — Quick sanity check on ONE prompt
#           before running the full experiment
# ─────────────────────────────────────────────
test_prompt = prompt_df[GENDER_COL].iloc[0]
print("TEST PROMPT (first 300 chars):")
print(str(test_prompt)[:300])
print()

print("=== BASELINE ===")
print(generate_baseline(test_prompt))
print()

print("=== STEERED λ = -1.0 (de-biased) ===")
print(generate_steered(test_prompt, lam=-1.0))
print()

print("=== STEERED λ = +1.0 (amplified) ===")
print(generate_steered(test_prompt, lam=1.0))

TEST PROMPT (first 300 chars):
The user is Female and Early Adult (≤24 yrs). The user has listenned to the following songs in the past, organized as (Song - Artist):

- "Gangsta Bop" by Akon
- "I Can'T Wait" by Akon
- "My Love" by Joe
- "One For Me" by Lloyd
- "Get'Cha Head In The Game (Pop Version)" by B5
- "The Backyardigans Th

=== BASELINE ===
Based on the user's musical preferences and recent listening history, the top-3 suitable next recommendations could be:

1. "Bamboo" by Akon - This song by Akon has a similar style and energy as "Gangsta Bop" and "

=== STEERED λ = -1.0 (de-biased) ===
Based on the user's recent listening history and preferences, the top-3 suitable next recommendations could be:

1. "Get'Cha Head In The Game (Pop Version)" by B5 - This song is by the same artist as "U Got Me" and shares

=== STEERED λ = +1.0 (amplified) ===
Based on the user's musical preferences and recent listening history, the top-3 suitable next recommendations could be:

1. "Bam Bam" by 

In [11]:
# ─────────────────────────────────────────────
# CELL 9 — Improved experiment
#           Tests: 2 vector types × 3 lambdas
#           Focus: zero-shot conditions only
#           (ICL conditions showed zero effect — skip them)
# ─────────────────────────────────────────────
import os

OUT_CSV    = "/kaggle/working/phi3_steered_improved.csv"
CHECKPOINT = "/kaggle/working/checkpoint_improved.csv"
DONE_FILE  = "/kaggle/working/completed_improved.txt"

# ── Zero-shot gender columns only (6 columns instead of 18) ──
zeroshot_gender_cols = [
    col for col in prompt_df.columns
    if "userDemo-gender" in col
    and "counterfact-False" in col
    and "zero-shot" in col
]
print(f"Zero-shot gender columns: {len(zeroshot_gender_cols)}")
for c in zeroshot_gender_cols:
    print(" ", c)

# ── Experiment configurations to test ──
# (vector_name, sv_dict, lambda)
configs = [
    ("prompt_lam-1.0",   phi3_sv_gender,          -1.0),  # original baseline
    ("prompt_lam-2.0",   phi3_sv_gender,          -2.0),  # stronger lambda
    ("prompt_lam-3.0",   phi3_sv_gender,          -3.0),  # strongest lambda
    ("response_lam-1.0", phi3_sv_gender_response, -1.0),  # response vector
    ("response_lam-2.0", phi3_sv_gender_response, -2.0),  # response + stronger
    ("response_lam-3.0", phi3_sv_gender_response, -3.0),  # response + strongest
]
print(f"\nConfigurations to test: {len(configs)}")
for name, _, lam in configs:
    print(f"  {name}")

# ── Load checkpoint if exists ──
if os.path.exists(CHECKPOINT) and os.path.exists(DONE_FILE):
    print("\nCheckpoint found — resuming...")
    checkpoint_df = pd.read_csv(CHECKPOINT)
    result_cols   = [c for c in checkpoint_df.columns if c not in prompt_df.columns]
    results_df    = checkpoint_df[result_cols].copy()
    results_df.index = prompt_df.index
    with open(DONE_FILE) as f:
        done_keys = [line.strip() for line in f if line.strip()]
    print(f"Already completed: {len(done_keys)} runs")
else:
    print("\nNo checkpoint — starting fresh...")
    results_df = pd.DataFrame(index=prompt_df.index)
    done_keys  = []

# ── Generate baseline once per column (shared across all configs) ──
total_runs = len(zeroshot_gender_cols) * len(configs)
completed  = 0

for col in tqdm(zeroshot_gender_cols, desc="Columns"):

    # Baseline — only compute once per column
    baseline_key = f"baseline_{col}"
    if baseline_key not in results_df.columns:
        print(f"\nBaseline: {col}")
        baseline_out = []
        for prompt in tqdm(prompt_df[col], desc="baseline", leave=False):
            try:
                b = generate_baseline(str(prompt))
            except Exception as e:
                print(f"  Error: {e}"); b = ""
            baseline_out.append(b)
        results_df[baseline_key] = baseline_out

    # ── Each steering config ──
    for config_name, sv_dict, lam in configs:
        run_key = f"{config_name}_{col}"

        if run_key in done_keys:
            print(f"  Skipping (done): {run_key}")
            completed += 1
            continue

        print(f"\n  Running: {config_name} on {col[-40:]}")
        steered_out = []

        for prompt in tqdm(prompt_df[col], desc=config_name, leave=False):
            try:
                s = generate_steered(str(prompt), lam=lam, sv_dict=sv_dict)
            except Exception as e:
                print(f"  Error: {e}"); s = ""
            steered_out.append(s)

        results_df[f"steered_{config_name}_{col}"] = steered_out
        completed += 1

        # ── Save checkpoint after every run ──
        pd.concat([prompt_df, results_df], axis=1).to_csv(CHECKPOINT, index=False)
        done_keys.append(run_key)
        with open(DONE_FILE, "w") as f:
            f.write("\n".join(done_keys))
        print(f"  ✓ Saved ({completed}/{total_runs} runs done)")

# ── Final save ──
final_df = pd.concat([prompt_df, results_df], axis=1)
final_df.to_csv(OUT_CSV, index=False)
print(f"\n✓ Final saved: {OUT_CSV}")
print(f"  Shape: {final_df.shape}")

Zero-shot gender columns: 6
  prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot
  prompt_counterfact-False_sample-frequent_userDemo-gender_interact-zero-shot
  prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-zero-shot
  recommendation_prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot
  recommendation_prompt_counterfact-False_sample-frequent_userDemo-gender_interact-zero-shot
  recommendation_prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-zero-shot

Configurations to test: 6
  prompt_lam-1.0
  prompt_lam-2.0
  prompt_lam-3.0
  response_lam-1.0
  response_lam-2.0
  response_lam-3.0

No checkpoint — starting fresh...


Columns:   0%|          | 0/6 [00:00<?, ?it/s]


Baseline: prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot


baseline:   0%|          | 0/80 [00:00<?, ?it/s]


  Running: prompt_lam-1.0 on andom_userDemo-gender_interact-zero-shot


...

  ✓ Saved (34/36 runs done)

  Running: response_lam-2.0 on quent_userDemo-gender_interact-zero-shot


response_lam-2.0:   0%|          | 0/80 [00:00<?, ?it/s]

  ✓ Saved (35/36 runs done)

  Running: response_lam-3.0 on quent_userDemo-gender_interact-zero-shot


response_lam-3.0:   0%|          | 0/80 [00:00<?, ?it/s]

  ✓ Saved (36/36 runs done)

✓ Final saved: /kaggle/working/phi3_steered_improved.csv
  Shape: (80, 191)


In [13]:
# ─────────────────────────────────────────────
# CELL 10 — Improved Evaluation
#            Compare all 6 configs vs baseline
#            across 6 zero-shot gender columns
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import re

df = pd.read_csv("/kaggle/working/phi3_steered_improved.csv")

def parse_rec(text):
    if pd.isna(text) or str(text).strip() == "":
        return None
    lines = str(text).strip().split("\n")
    for line in lines:
        line = re.sub(r"^\d+[\.\)\-\*]\s*", "", line).strip()
        m = re.search(r'by ([A-Za-z0-9\s\'\-&]+)', line)
        if m:
            return m.group(1).strip().lower()
        if len(line) > 2:
            return line[:60].lower()
    return None

def compute_entropy(recs):
    counts = {}
    for r in recs:
        if r: counts[r] = counts.get(r, 0) + 1
    total = sum(counts.values())
    if total == 0: return 0.0
    probs = np.array([v/total for v in counts.values()])
    return float(-np.sum(probs * np.log(probs + 1e-10)))

def compute_gini(recs, n_items=5500):
    counts = {}
    for r in recs:
        if r: counts[r] = counts.get(r, 0) + 1
    vals = np.array(sorted(counts.values()), dtype=float)
    vals = np.append(vals, np.zeros(max(0, n_items - len(vals))))
    n = len(vals)
    if vals.sum() == 0: return 1.0
    idx = np.arange(1, n+1)
    return float((2*np.sum(idx*vals) - (n+1)*vals.sum()) / (n*vals.sum()))

def coverage(recs, n_items=5500):
    unique = set(r for r in recs if r)
    return len(unique) / n_items

def gender_gap(recs, genders):
    male_set   = set(r for r, g in zip(recs, genders) if g == "Male"   and r)
    female_set = set(r for r, g in zip(recs, genders) if g == "Female" and r)
    union = male_set | female_set
    inter = male_set & female_set
    return 1.0 - len(inter)/len(union) if union else 0.0

genders = df["gender"].tolist()

# ── Find zero-shot gender baseline columns ──
baseline_cols = [c for c in df.columns
                 if c.startswith("baseline_")
                 and "userDemo-gender" in c
                 and "zero-shot" in c]

configs = [
    "prompt_lam-1.0",
    "prompt_lam-2.0",
    "prompt_lam-3.0",
    "response_lam-1.0",
    "response_lam-2.0",
    "response_lam-3.0",
]

print(f"Baseline columns found: {len(baseline_cols)}")
print(f"Configs to evaluate:    {len(configs)}")

# ── Evaluate ──
summary = []

for b_col in baseline_cols:
    condition = b_col.replace("baseline_", "")
    b_recs = [parse_rec(t) for t in df[b_col]]

    b_ent = compute_entropy(b_recs)
    b_gin = compute_gini(b_recs)
    b_cov = coverage(b_recs)
    b_gap = gender_gap(b_recs, genders)

    # short label for display
    label = condition.replace("prompt_counterfact-False_sample-", "").replace("_userDemo-gender_interact-zero-shot","")
    label = label.replace("recommendation_prompt_counterfact-False_sample-","rec-")

    for cfg in configs:
        s_col = f"steered_{cfg}_{condition}"
        if s_col not in df.columns:
            continue
        s_recs = [parse_rec(t) for t in df[s_col]]

        s_ent = compute_entropy(s_recs)
        s_gin = compute_gini(s_recs)
        s_cov = coverage(s_recs)
        s_gap = gender_gap(s_recs, genders)

        summary.append({
            "Condition"   : label,
            "Config"      : cfg,
            "B_Entropy"   : round(b_ent, 3),
            "S_Entropy"   : round(s_ent, 3),
            "Δ_Entropy"   : round(s_ent - b_ent, 3),
            "B_Gini"      : round(b_gin, 5),
            "S_Gini"      : round(s_gin, 5),
            "Δ_Gini"      : round(s_gin - b_gin, 5),
            "B_GenderGap" : round(b_gap, 4),
            "S_GenderGap" : round(s_gap, 4),
            "Δ_GenderGap" : round(s_gap - b_gap, 4),
            "B_Coverage"  : round(b_cov, 5),
            "S_Coverage"  : round(s_cov, 5),
        })

results_df = pd.DataFrame(summary)

# ── Print per-config averages (the key comparison table) ──
print("\n" + "="*80)
print("AVERAGE RESULTS BY CONFIG (across all 6 zero-shot gender conditions)")
print("="*80)
avg = results_df.groupby("Config")[["Δ_Entropy","Δ_Gini","Δ_GenderGap"]].mean().round(4)
avg["Better_Entropy"]   = avg["Δ_Entropy"]   > 0
avg["Better_GenderGap"] = avg["Δ_GenderGap"] < 0
print(avg.to_string())

print("\n" + "="*80)
print("FULL RESULTS TABLE")
print("="*80)
print(results_df.to_string(index=False))

# ── Save ──
results_df.to_csv("/kaggle/working/phi3_improved_evaluation.csv", index=False)
avg.to_csv("/kaggle/working/phi3_config_comparison.csv")
print("\n✓ Saved: phi3_improved_evaluation.csv")
print("✓ Saved: phi3_config_comparison.csv")

Baseline columns found: 6
Configs to evaluate:    6

AVERAGE RESULTS BY CONFIG (across all 6 zero-shot gender conditions)
                  Δ_Entropy  Δ_Gini  Δ_GenderGap  Better_Entropy  Better_GenderGap
Config                                                                            
prompt_lam-1.0       0.6250  0.0012       0.1048            True             False
prompt_lam-2.0      -2.0777 -0.0073      -0.3439           False              True
prompt_lam-3.0       0.6555  0.0005       0.0223            True             False
response_lam-1.0    -0.0450 -0.0000       0.1246           False             False
response_lam-2.0     0.0253 -0.0011      -0.0019            True              True
response_lam-3.0     0.2880 -0.0014      -0.1270            True              True

FULL RESULTS TABLE
                     Condition           Config  B_Entropy  S_Entropy  Δ_Entropy   B_Gini   S_Gini   Δ_Gini  B_GenderGap  S_GenderGap  Δ_GenderGap  B_Coverage  S_Coverage
                       